# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

Sources:

https://www.geeksforgeeks.org/pandas/how-to-count-distinct-values-of-a-pandas-dataframe-column/

Google Gemini: I used Google Gemini to help me understand syntax errors I was getting but also I used it to help me understand the question properly to help me make sure I was answering it correctly.

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
product_containing_poptart = cat[cat['product_name'].str.lower().str.contains('pop')]
print(f"Count: {len(product_containing_poptart)}")
print("Rows:")
product_containing_poptart

Count: 4
Rows:


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
print("SKU and names that all mean strawberry Pop-Tart:")
product_containing_strawberry_poptart = product_containing_poptart[product_containing_poptart['product_name'].str.lower().str.contains('strawberry')]
print(product_containing_strawberry_poptart[['sku','product_name']])

SKU and names that all mean strawberry Pop-Tart:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry


**Pop-Tart rows that are a different product:** No rows of Pop-Tart are a different product as all of the rows contain the product word of strawberry in them

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
CANON = {
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

def consolidate(df):
    out = df.copy()
    out['canonical_sku'] = out['sku'].map(CANON).fillna(out['sku'])
    return out

consolidated = consolidate(cat)
print(consolidated.head())

             sku          product_name            category subcategory  \
0          PT-12  pop-tarts strawberry     Breakfast Foods         NaN   
1     SKU#459812   Strawberry Pop-Tart           Breakfast   Pop-Tarts   
2  POPTART-STRAW  Pop-Tarts Strawberry           Breakfast         NaN   
3  PT_STRAWBERRY  pop-tarts strawberry           breakfast   Pop-Tarts   
4         WTR-24    BOTTLED WATER 24PK  Emergency Supplies     Bottled   

   unit_cost           supplier package_size  canonical_sku  
0       1.93  Johnson & Johnson       4-pack  POPTART-STRAW  
1       2.26           Roku Inc          4pk  POPTART-STRAW  
2       2.11           Sara Lee       6 pack  POPTART-STRAW  
3        NaN         Campbell's       4 pack  POPTART-STRAW  
4       2.64           Sara Lee       1 pack         WTR-24  


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print("Distinct category before strip and title-case:")
print(consolidated['category'].value_counts())
consolidated['category_normalized'] = consolidated['category'].str.strip().str.title()
print("Distinct category after strip and title-case:")
print(consolidated['category_normalized'].value_counts())

Distinct category before strip and title-case:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
Distinct category after strip and title-case:
category_normalized
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Ba

The pairs that are still two label for one thing are: Emergency and Emergency Supplies, Baby and Baby Products, Alcohol and Alcoholic Beverages, Breakfast and Breakfast Foods, and Grocery and Groceries.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
cat['normalized'] = (cat['product_name'].str.lower()
                         .str.replace(r'[^a-z ]', '', regex=True)
                         .str.split().apply(lambda w: ' '.join(sorted(w))))
candidates = (cat.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

,sku
normalized,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
# TODO
assert consolidated['canonical_sku'].isnull().any() != True, 'Canonical column should not have any nulls'
assert len(consolidated['canonical_sku']) == len(cat), 'Canonical rows should not change the number of rows'

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families beyond Pop-Tarts that appear fragmented are bottled water and batteries. I would group them based on the sku such as for water. Instead of being three different types I would make it based on one single sku such as WATER_24PK and the same for the batteries such as listing them all as BAT-AA4.

b) I think where consolidation stops being a data-cleaning task and becomes a business decision is when we consolidate sku for different units. An example would be AA batteries. There are three separate AA battery units all with different package_sizes. By consolidating the sku to all the same it ignores the package_size difference which can lead to some incorrect business decisions. I believe asking someone with higher authority such as a manager or a distribution coordinator would help to alleviate any potential issues that consolidation might cause.
